# AWS Stock Agent — Acceptance Notebook

This notebook is organized so that each major section maps directly to a customer acceptance criterion.

## What this notebook demonstrates
1. Environment and deployment parameters
2. Cognito authentication
3. Invocation of the deployed AWS AgentCore runtime
4. Execution of the five required acceptance queries
5. Evidence placeholders for streaming behavior
6. Evidence placeholders for Langfuse traces
7. A final acceptance summary

> Replace placeholders only where marked. Do **not** hardcode secrets into the notebook before sharing.

## 0. Prerequisites

Before running the notebook, confirm the following are available to the reviewer:

- AWS credentials with permission to invoke the AgentCore runtime
- A valid Cognito test user
- OpenAI-backed runtime already deployed
- Langfuse project configured
- The knowledge base ingested with:
  - Amazon 2024 Annual Report
  - AMZN Q3 2025 Earnings Release
  - AMZN Q2 2025 Earnings Release

This notebook is meant to be executable by the review team with minimal edits.

In [ ]:
# Optional: install dependencies if the review environment is clean.
# Uncomment if needed.

# %pip install boto3 requests pandas matplotlib pillow

## 1. Deployment parameters

This section maps to:
- Source code in a repository with clear deployment documentation
- Executable notebook against the deployed endpoint
- Cognito user pool configuration

In [ ]:
from pathlib import Path
import os
import json
import uuid
import time
from datetime import datetime

AWS_REGION = "us-east-1"

# Replace only if your runtime ARN changes.
AGENT_RUNTIME_ARN = "arn:aws:bedrock-agentcore:us-east-1:148761635572:runtime/aws_stock_agent_dev_runtime-po9K3zBMok"

# Cognito values from Terraform outputs
COGNITO_USER_POOL_ID = "us-east-1_4iYUB1Kgh"
COGNITO_CLIENT_ID = "2c2d618m3n35rep3904bhlkfnk"
COGNITO_CLIENT_SECRET = ""  # optional; fill only if your flow requires it
COGNITO_DOMAIN_PREFIX = "stock-agent-dev-auth"

# Reviewer-provided test credentials
COGNITO_USERNAME = os.getenv("COGNITO_USERNAME", "")
COGNITO_PASSWORD = os.getenv("COGNITO_PASSWORD", "")

# Optional Langfuse evidence file. If you export a screenshot locally, point to it here.
LANGFUSE_SCREENSHOT_PATH = Path("langfuse_trace_screenshot.png")

# Output folder for saved responses
OUTPUT_DIR = Path("acceptance_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print({
    "AWS_REGION": AWS_REGION,
    "AGENT_RUNTIME_ARN": AGENT_RUNTIME_ARN,
    "COGNITO_USER_POOL_ID": COGNITO_USER_POOL_ID,
    "COGNITO_CLIENT_ID": COGNITO_CLIENT_ID,
    "OUTPUT_DIR": str(OUTPUT_DIR.resolve()),
})

## 2. Cognito authentication

This section maps to:
- Setup Cognito user pool for inbound user authorization
- Notebook should show user authentication from Cognito user pool

If your Cognito app client uses a client secret, populate `COGNITO_CLIENT_SECRET`.
If not, leave it blank.

In [ ]:
import base64
import boto3
import hashlib
import hmac

def compute_secret_hash(username: str, client_id: str, client_secret: str) -> str:
    message = username + client_id
    digest = hmac.new(
        client_secret.encode("utf-8"),
        message.encode("utf-8"),
        hashlib.sha256,
    ).digest()
    return base64.b64encode(digest).decode()

def cognito_authenticate():
    if not COGNITO_USERNAME or not COGNITO_PASSWORD:
        raise ValueError("Set COGNITO_USERNAME and COGNITO_PASSWORD before running this cell.")

    client = boto3.client("cognito-idp", region_name=AWS_REGION)
    auth_parameters = {
        "USERNAME": COGNITO_USERNAME,
        "PASSWORD": COGNITO_PASSWORD,
    }

    if COGNITO_CLIENT_SECRET:
        auth_parameters["SECRET_HASH"] = compute_secret_hash(
            COGNITO_USERNAME,
            COGNITO_CLIENT_ID,
            COGNITO_CLIENT_SECRET,
        )

    response = client.initiate_auth(
        ClientId=COGNITO_CLIENT_ID,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters=auth_parameters,
    )
    return response

auth_response = None
try:
    auth_response = cognito_authenticate()
    redacted = {
        "AuthenticationResult": {
            "AccessToken": "***redacted***" if auth_response.get("AuthenticationResult", {}).get("AccessToken") else None,
            "IdToken": "***redacted***" if auth_response.get("AuthenticationResult", {}).get("IdToken") else None,
            "RefreshToken": "***redacted***" if auth_response.get("AuthenticationResult", {}).get("RefreshToken") else None,
            "TokenType": auth_response.get("AuthenticationResult", {}).get("TokenType"),
            "ExpiresIn": auth_response.get("AuthenticationResult", {}).get("ExpiresIn"),
        },
        "ChallengeName": auth_response.get("ChallengeName"),
    }
    print(json.dumps(redacted, indent=2))
except Exception as e:
    print("Cognito auth not completed:", str(e))
    print("If the runtime is not yet protected by Cognito, keep this section as infrastructure/auth evidence.")

## 3. Runtime invocation helper

This section maps to:
- AgentCore runtime hosted via FastAPI
- Executable notebook against deployed endpoint

The helper below invokes the deployed AgentCore runtime through AWS CLI because that is already validated in the project workflow.

In [ ]:
import subprocess
import shlex

def invoke_runtime_cli(query: str, session_id: str | None = None, extra_payload: dict | None = None):
    session_id = session_id or f"session-{uuid.uuid4().hex[:12]}"
    payload = {
        "message": query,
        "user_id": "notebook-reviewer",
        "session_id": session_id,
    }
    if extra_payload:
        payload.update(extra_payload)

    payload_path = OUTPUT_DIR / f"{session_id}_payload.json"
    response_path = OUTPUT_DIR / f"{session_id}_response.json"

    payload_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

    cmd = [
        "aws", "bedrock-agentcore", "invoke-agent-runtime",
        "--region", AWS_REGION,
        "--agent-runtime-arn", AGENT_RUNTIME_ARN,
        "--content-type", "application/json",
        "--accept", "application/json",
        "--runtime-session-id", session_id,
        "--payload", f"fileb://{payload_path}",
        str(response_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    return {
        "session_id": session_id,
        "payload_path": str(payload_path),
        "response_path": str(response_path),
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "response_json": json.loads(response_path.read_text(encoding="utf-8")) if response_path.exists() else None,
    }

# Quick smoke test for the deployed runtime
smoke = invoke_runtime_cli("What is the stock price for Amazon right now?")
print(json.dumps(smoke["response_json"], indent=2))

## 4. Acceptance query 1 — current stock price

Customer criterion:
- **What is the stock price for Amazon right now?**

In [ ]:
query_1 = "What is the stock price for Amazon right now?"
result_1 = invoke_runtime_cli(query_1)
print(json.dumps(result_1["response_json"], indent=2))

## 5. Acceptance query 2 — historical stock price in Q4 last year

Customer criterion:
- **What were the stock prices for Amazon in Q4 last year?**

This is the key proof point for `retrieve_historical_stock_price`.

In [ ]:
query_2 = "What were the stock prices for Amazon in Q4 last year?"
result_2 = invoke_runtime_cli(query_2)
print(json.dumps(result_2["response_json"], indent=2))

## 6. Acceptance query 3 — stock performance vs analysts' reported expectations

Customer criterion:
- **Compare Amazon's recent stock performance to what analysts predicted in their reports**

This query should hit the knowledge base built from the Amazon reports.

In [ ]:
query_3 = "Compare Amazon's recent stock performance to what analysts predicted in their reports"
result_3 = invoke_runtime_cli(query_3)
print(json.dumps(result_3["response_json"], indent=2))

## 7. Acceptance query 4 — current price plus AI business context

Customer criterion:
- **I’m researching AMZN give me the current price and any relevant information about their AI business**

In [ ]:
query_4 = "I’m researching AMZN give me the current price and any relevant information about their AI business"
result_4 = invoke_runtime_cli(query_4)
print(json.dumps(result_4["response_json"], indent=2))

## 8. Acceptance query 5 — office space in North America in 2024

Customer criterion:
- **What is the total amount of office space Amazon owned in North America in 2024?**

This is the strongest retrieval test against the Annual Report.

In [ ]:
query_5 = "What is the total amount of office space Amazon owned in North America in 2024?"
result_5 = invoke_runtime_cli(query_5)
print(json.dumps(result_5["response_json"], indent=2))

## 9. Consolidated response table

This section helps the reviewer see all five acceptance outputs at once.

In [ ]:
import pandas as pd

rows = []
for idx, item in enumerate([result_1, result_2, result_3, result_4, result_5], start=1):
    rows.append({
        "query_number": idx,
        "session_id": item["session_id"],
        "returncode": item["returncode"],
        "reply": (item["response_json"] or {}).get("reply"),
        "status": (item["response_json"] or {}).get("status"),
    })

df = pd.DataFrame(rows)
df

## 10. Streaming evidence placeholder

Customer criterion:
- **Event responses must be streamed**
- **Streams events via `.astream()`**

If the deployed endpoint already returns streamed events, capture raw event output here.
If streaming is still exposed through a different path than the CLI invoke command, replace this cell with the correct client call.

This notebook includes a placeholder so the reviewer can attach evidence from the streamed response path.

In [ ]:
streaming_evidence = {
    "status": "TODO / replace with actual streaming capture",
    "notes": [
        "If using HTTP chunked response or SSE, capture each event chunk here.",
        "If using a dedicated stream endpoint, replace this placeholder with the real call.",
        "If using the same runtime but a different client, paste the event sequence output here."
    ]
}
print(json.dumps(streaming_evidence, indent=2))

## 11. Langfuse evidence

Customer criterion:
- **Setup Langfuse cloud free tier for observability**
- **Notebook should contain screenshots or API response showing Langfuse traces**

This notebook supports the screenshot route because that is the safest artifact for a customer handoff.
Export a trace screenshot from Langfuse and place it next to this notebook as `langfuse_trace_screenshot.png`.

In [ ]:
if LANGFUSE_SCREENSHOT_PATH.exists():
    from PIL import Image
    display(Image.open(LANGFUSE_SCREENSHOT_PATH))
else:
    print("Langfuse screenshot not found.")
    print(f"Expected file: {LANGFUSE_SCREENSHOT_PATH.resolve()}")
    print("Add a screenshot export from Langfuse here before final delivery.")

## 12. Optional raw outputs archive

This section saves all responses for handoff or audit.

In [ ]:
archive = {
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
    "queries": {
        "q1_current_price": result_1["response_json"],
        "q2_historical_q4": result_2["response_json"],
        "q3_analyst_comparison": result_3["response_json"],
        "q4_ai_business": result_4["response_json"],
        "q5_office_space": result_5["response_json"],
    }
}
archive_path = OUTPUT_DIR / "acceptance_summary.json"
archive_path.write_text(json.dumps(archive, indent=2), encoding="utf-8")
print(f"Saved: {archive_path.resolve()}")

## 13. Final acceptance checklist

Mark each item after validation.

In [ ]:
acceptance_checklist = [
    {"criterion": "Repository and README available", "status": "DONE / REVIEW"},
    {"criterion": "Runtime deployed on AWS AgentCore", "status": "DONE / REVIEW"},
    {"criterion": "Cognito authentication demonstrated", "status": "PENDING REVIEW"},
    {"criterion": "Langfuse traces demonstrated", "status": "PENDING REVIEW"},
    {"criterion": "Realtime stock query executed", "status": "DONE / REVIEW"},
    {"criterion": "Historical stock query executed", "status": "DONE / REVIEW"},
    {"criterion": "Report comparison query executed", "status": "DONE / REVIEW"},
    {"criterion": "AI business query executed", "status": "DONE / REVIEW"},
    {"criterion": "Office space retrieval query executed", "status": "DONE / REVIEW"},
    {"criterion": "Streaming evidence attached", "status": "PENDING REVIEW"},
]
pd.DataFrame(acceptance_checklist)